# Rategoan — Panen Dataset (Kaggle, API resmi)

Port dari ide `colab/panen-korpus.ipynb` (Mode A/crawl situs sudah dihapus —
terbukti rapuh, tergantung struktur HTML tiap situs yang bisa berubah
kapan saja). Notebook ini **cuma Mode B — dataset aggregator lewat
HuggingFace `datasets` streaming**, tidak ada crawl situs sama sekali.
Notebook training `kaggle/latih-gpu.ipynb` dan `colab/latih-gpu.ipynb`
tidak disentuh.

**Token tidak ditulis di notebook.** Secret Kaggle (Add-ons > Secrets),
nama persis:
- `GITHUB_TOKEN` — PAT repo (scope `repo`), wajib
- `HF_TOKEN` — token HuggingFace, opsional (hanya untuk dataset gated
  seperti CulturaX; tanpa ini dataset gated dilewati dengan peringatan)

## 2 LANGKAH (yang lain otomatis)

1. Add-ons > Secrets di Kaggle, tambahkan `GITHUB_TOKEN` (+ `HF_TOKEN`
   kalau mau panen CulturaX). Aktifkan akses notebook ini ke kedua secret.
2. Cek sel "Pilih dataset + anggaran panen" di bawah (nilai default sudah
   aman untuk uji coba), lalu **Save & Run All** (atau
   `kaggle kernels push -p kaggle/panen-dataset` lewat `kernel-metadata.json`
   di folder ini — jalur resmi [Kaggle Kernels API](https://github.com/Kaggle/kaggle-api/blob/main/docs/kernels.md)).
   Folder ini terpisah dari `kaggle/latih-gpu.ipynb` + `kaggle/kernel-metadata.json`
   (notebook training, tidak disentuh) — Kaggle API butuh satu
   `kernel-metadata.json` per folder kernel.

Kernel mati/quota habis di tengah panen? Push/run lagi kernel ini —
progres per dataset otomatis diunduh balik dari Release
(`panen-<dataset>` → asset `progress.json`), offset yang sudah
dikonsumsi tidak diulang.

## 4 dataset (semua CC0/CC-BY, tidak ada crawl situs)

| Dataset | Lisensi | Peran |
|---|---|---|
| **MADLAD-400 id** | CC-BY-4.0 | **lumbung utama** — anggaran default 3x dataset lain |
| OSCAR id | CC-BY-SA-4.0 | pelengkap volume |
| CulturaX id | campuran (mC4+OSCAR terkurasi), gated | pelengkap volume, perlu `HF_TOKEN` |
| Indo4B | CC0 | pelengkap volume |

Hasil di-upload ke tag Release **`panen-<dataset>`** (staging) — BUKAN
tag kanonik `korpus-<kategori>-bersih`. Masuk korpus training resmi tetap
lewat review manual sesuai `PRD-DATA-RELEASE.md` §5 (dedupe lintas-file,
klasifikasi kategori K1/K2/K3, hanya dirigen/Grok yang publish ke tag
kanonik) — PRD tidak diubah oleh notebook ini.

In [ ]:
# @title 1. Pilih dataset + anggaran panen (satu-satunya nilai manual)
import os

# MODE_PANEN: "SEMUA" atau salah satu slug -> "madlad400-id" | "oscar-id" | "culturax-id" | "indo4b"
# (Kaggle tidak punya widget dropdown Colab @param - ganti nilai di sel ini secara
# langsung, atau timpa lewat env var RAGET_MODE_PANEN kalau menjalankan tanpa buka UI)
MODE_PANEN = os.environ.get('RAGET_MODE_PANEN', 'SEMUA')
MAKS_DOKUMEN_PER_DATASET = int(os.environ.get('RAGET_MAKS_DOKUMEN', '60000'))
MAKS_MENIT_PER_DATASET = int(os.environ.get('RAGET_MAKS_MENIT', '12'))
PENGALI_LUMBUNG_UTAMA = 3  # MADLAD-400 id dapat anggaran 3x dataset lain (lihat markdown di atas)

print('Mode panen:', MODE_PANEN)
print('Anggaran dasar per dataset: maks', MAKS_DOKUMEN_PER_DATASET, 'dokumen atau',
      MAKS_MENIT_PER_DATASET, 'menit (mana lebih dulu tercapai).')
print('MADLAD-400 id (lumbung utama) dapat', PENGALI_LUMBUNG_UTAMA, 'x anggaran itu.')

In [ ]:
# @title 2. Ambil GITHUB_TOKEN (wajib) + HF_TOKEN (opsional, dataset gated) dari Kaggle Secrets
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
GITHUB_TOKEN = _secrets.get_secret('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise SystemExit('Tambahkan secret Kaggle bernama GITHUB_TOKEN. Jangan tempel token di sel.')
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

try:
    HF_TOKEN = _secrets.get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

OWNER, REPO = 'maetalizer-png', 'Rategoan'
print('GITHUB_TOKEN OK, panjang', len(GITHUB_TOKEN), '(nilai tidak dicetak)')
print('HF_TOKEN:', 'ADA (dataset gated bisa diakses)' if HF_TOKEN else 'tidak diset (dataset gated akan dilewati)')

In [ ]:
# @title 3. Clone/pull repo (token disisipkan di URL - tidak tercetak ke log)
import subprocess

url = 'https://x-access-token:' + GITHUB_TOKEN + '@github.com/' + OWNER + '/' + REPO + '.git'
if os.path.isdir('Rategoan/.git'):
    subprocess.check_call(['git', '-C', 'Rategoan', 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--depth', '1', url, 'Rategoan'])
os.chdir('Rategoan')
print('repo OK, cwd:', os.getcwd())

In [ ]:
# @title 4. Pasang dependensi panen (cuma dataset streaming, tanpa crawler)
import subprocess, sys
pkgs = ['datasets>=2.19', 'huggingface_hub']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, capture_output=True, text=True)
print(r.stdout[-1000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit('Gagal pasang dependensi.')
import datasets as hf_datasets
print('datasets', hf_datasets.__version__, 'OK')

In [ ]:
# @title 5. Filter bahasa Indonesia (PRD Sec1.1 poin 5) + pembersih teks
import re, hashlib

STOPWORD_ID = set("""
yang dan di ke dari ini itu tidak dengan untuk pada akan adalah dalam oleh
sebagai juga atau karena namun sudah telah bisa dapat para lebih tersebut
tahun kata orang jika saat setelah sebelum antara tentang seperti bagi
""".split())

def rasio_stopword_id(teks):
    kata = re.findall(r"[a-zA-Z]+", teks.lower())
    if len(kata) < 20:
        return 0.0
    cocok = sum(1 for k in kata if k in STOPWORD_ID)
    return cocok / len(kata)

def cukup_indonesia(teks, ambang=0.06):
    """Pengaman tambahan - dataset sudah dipilih konfigurasi lang=id, tapi
    tetap dicek ulang di sini (bukan dari HTML mentah, dari field text yang
    sudah diekstrak) sesuai pelajaran keputusan-014: label sumber tidak
    selalu bisa dipercaya buta."""
    return rasio_stopword_id(teks) >= ambang

_ws_re = re.compile(r"[ \t\u00a0]+")
_ctrl_re = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def bersihkan_teks(teks):
    teks = _ctrl_re.sub("", teks)
    teks = _ws_re.sub(" ", teks)
    baris = [b.strip() for b in teks.split("\n")]
    baris = [b for b in baris if b]
    return "\n".join(baris).strip()

def hash_dedup(teks):
    return hashlib.sha1(teks.strip().lower().encode("utf-8")).hexdigest()

print("Filter bahasa + pembersih teks + hash dedup siap.")

In [ ]:
# @title 6. WHITELIST dataset Mode B (HuggingFace, streaming, lang=id)
WHITELIST_MODE_B = [
    dict(slug="madlad400-id", nama="MADLAD-400 id (lumbung utama)", hf_id="allenai/madlad-400", config="id",
         split="clean", field_text="text", lisensi="CC-BY-4.0", gated=False, lumbung_utama=True),
    dict(slug="oscar-id", nama="OSCAR id", hf_id="oscar-corpus/OSCAR-2301", config="id",
         split="train", field_text="text", lisensi="CC-BY-SA-4.0", gated=False, lumbung_utama=False),
    dict(slug="culturax-id", nama="CulturaX id", hf_id="uonlp/CulturaX", config="id",
         split="train", field_text="text", lisensi="Campuran (mC4+OSCAR terkurasi) - lihat kartu dataset",
         gated=True, lumbung_utama=False),
    dict(slug="indo4b", nama="Indo4B", hf_id="indobenchmark/indo4b", config=None,
         split="train", field_text="text", lisensi="CC0", gated=False, lumbung_utama=False),
]

if MODE_PANEN != "SEMUA":
    WHITELIST_MODE_B = [d for d in WHITELIST_MODE_B if d["slug"] == MODE_PANEN]
    if not WHITELIST_MODE_B:
        raise SystemExit(
            'MODE_PANEN "{}" tidak dikenal. Pilihan: SEMUA, madlad400-id, oscar-id, culturax-id, indo4b.'.format(MODE_PANEN)
        )

print("Dataset dijalankan sesi ini:", [d["slug"] for d in WHITELIST_MODE_B])
print("CATATAN: id HuggingFace di atas adalah rujukan terbaik yang diketahui saat notebook ditulis.")
print("Kalau salah satu gagal resolve (dataset dipindah/di-rename di HF), sel panen mencetak")
print("peringatan dan MELEWATI dataset itu, bukan menghentikan seluruh panen.")

In [ ]:
# @title 7. Util Release (auto-resume + upload) - REST API langsung, tanpa dependensi ekstra
import json, urllib.request, urllib.error

API = "https://api.github.com/repos/{}/{}".format(OWNER, REPO)

def _api(method, path, token, body=None, timeout=60):
    url = path if path.startswith("http") else API + path
    headers = {"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json"}
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read() or b"{}"
            return resp.status, json.loads(raw)
    except urllib.error.HTTPError as e:
        raw = e.read() or b"{}"
        try:
            parsed = json.loads(raw)
        except Exception:
            parsed = {"raw": raw.decode("utf-8", "replace")}
        return e.code, parsed

def get_or_create_release(tag, title=None, body="Panen dataset staging - lihat manifest.json. BUKAN tag kanonik."):
    st, rel = _api("GET", "/releases/tags/" + tag, GITHUB_TOKEN)
    if st == 404:
        st, rel = _api("POST", "/releases", GITHUB_TOKEN, {
            "tag_name": tag, "name": title or tag, "body": body, "draft": False, "prerelease": False,
        })
        if st not in (200, 201):
            raise RuntimeError("Gagal buat release {}: {}".format(tag, rel))
    elif st != 200:
        raise RuntimeError("Gagal cek release {}: {}".format(tag, rel))
    return rel

def unduh_asset(tag, nama_asset, tujuan_lokal):
    st, rel = _api("GET", "/releases/tags/" + tag, GITHUB_TOKEN)
    if st != 200:
        return False
    asset = next((a for a in rel.get("assets", []) if a["name"] == nama_asset), None)
    if not asset:
        return False
    req = urllib.request.Request(asset["url"], headers={
        "Authorization": "Bearer " + GITHUB_TOKEN,
        "Accept": "application/octet-stream",
    })
    try:
        with urllib.request.urlopen(req, timeout=120) as resp, open(tujuan_lokal, "wb") as f:
            f.write(resp.read())
        return True
    except Exception as e:
        print("  gagal unduh asset resume", nama_asset, ":", e)
        return False

def unggah_asset(tag, nama_asset, path_lokal, title=None):
    rel = get_or_create_release(tag, title=title)
    existing = next((a for a in rel.get("assets", []) if a["name"] == nama_asset), None)
    if existing:
        _api("DELETE", "/releases/assets/{}".format(existing["id"]), GITHUB_TOKEN)
        _, rel = _api("GET", "/releases/" + str(rel["id"]), GITHUB_TOKEN)
    upload_url = rel["upload_url"].split("{")[0] + "?name=" + nama_asset
    with open(path_lokal, "rb") as f:
        data = f.read()
    req = urllib.request.Request(upload_url, data=data, method="POST", headers={
        "Authorization": "Bearer " + GITHUB_TOKEN,
        "Content-Type": "application/octet-stream",
        "Accept": "application/vnd.github+json",
    })
    with urllib.request.urlopen(req, timeout=1800) as resp:
        return json.loads(resp.read())

def muat_progres(slug, default):
    tag = "panen-" + slug
    lokal = "/kaggle/working/_progres_{}.json".format(slug)
    if unduh_asset(tag, "progress.json", lokal):
        try:
            with open(lokal, encoding="utf-8") as f:
                progres = json.load(f)
            print("  resume:", slug, "-", progres.get("ringkasan", "progres sebelumnya ditemukan"))
            return progres
        except Exception:
            pass
    return default

def simpan_progres(slug, progres):
    tag = "panen-" + slug
    lokal = "/kaggle/working/_progres_{}.json".format(slug)
    with open(lokal, "w", encoding="utf-8") as f:
        json.dump(progres, f, ensure_ascii=False)
    try:
        unggah_asset(tag, "progress.json", lokal)
    except Exception as e:
        print("  peringatan: gagal unggah progress.json ke", tag, ":", e)

print("Util Release (auto-resume + upload) siap.")

In [ ]:
# @title 8. Jalankan panen (streaming + filter lang=id + bersihkan + dedup, budget per dataset)
import time
from datasets import load_dataset

def panen_dataset(d, laporan_sumber):
    slug = d["slug"]
    if d.get("gated") and not HF_TOKEN:
        print("- LEWATI (gated, HF_TOKEN tidak diset):", d["nama"])
        laporan_sumber[slug] = {"nama": d["nama"], "status": "dilewati (gated, tanpa HF_TOKEN)", "dokumen": 0}
        return []

    batas_dokumen = MAKS_DOKUMEN_PER_DATASET * (PENGALI_LUMBUNG_UTAMA if d.get("lumbung_utama") else 1)
    batas_menit = MAKS_MENIT_PER_DATASET * (PENGALI_LUMBUNG_UTAMA if d.get("lumbung_utama") else 1)

    progres = muat_progres(slug, {"offset": 0, "diambil": 0, "ringkasan": ""})
    offset = progres["offset"]
    diambil_total_akumulasi = progres["diambil"]

    try:
        ds = load_dataset(d["hf_id"], d["config"], split=d["split"], streaming=True,
                           token=HF_TOKEN if d.get("gated") else None)
    except Exception as e:
        print("- GAGAL memuat dataset", d["nama"], "(", d["hf_id"], "):", e)
        print("  Kemungkinan id/config HF berubah - cek https://huggingface.co/datasets/" + d["hf_id"])
        laporan_sumber[slug] = {"nama": d["nama"], "status": "gagal dimuat: " + str(e)[:200], "dokumen": 0}
        return []

    if offset:
        ds = ds.skip(offset)

    hasil = []
    dilihat_hash = set()
    dibuang_bahasa = 0
    dibuang_duplikat = 0
    t_mulai = time.time()

    for contoh in ds:
        if len(hasil) >= batas_dokumen:
            break
        if (time.time() - t_mulai) / 60.0 >= batas_menit:
            break
        mentah = (contoh.get(d["field_text"]) or "").strip()
        offset += 1
        if len(mentah) < 200:
            continue
        teks = bersihkan_teks(mentah)
        if len(teks) < 200:
            continue
        if not cukup_indonesia(teks):
            dibuang_bahasa += 1
            continue
        h = hash_dedup(teks)
        if h in dilihat_hash:
            dibuang_duplikat += 1
            continue
        dilihat_hash.add(h)
        hasil.append({"text": teks, "source": d["nama"], "url": "hf://" + d["hf_id"], "license": d["lisensi"], "lang": "id"})
        if len(hasil) % 2000 == 0:
            simpan_progres(slug, {"offset": offset, "diambil": diambil_total_akumulasi + len(hasil),
                                   "ringkasan": "{} dokumen terkumpul".format(diambil_total_akumulasi + len(hasil))})

    simpan_progres(slug, {"offset": offset, "diambil": diambil_total_akumulasi + len(hasil),
                           "ringkasan": "{} dokumen terkumpul (sesi ini selesai)".format(diambil_total_akumulasi + len(hasil))})
    laporan_sumber[slug] = {
        "nama": d["nama"], "status": "selesai", "dokumenSesiIni": len(hasil),
        "totalAkumulasi": diambil_total_akumulasi + len(hasil),
        "dibuangBukanIndonesia": dibuang_bahasa, "dibuangDuplikat": dibuang_duplikat,
        "detikDipakai": round(time.time() - t_mulai, 1),
    }
    print("- {}: {} dokumen sesi ini (total akumulasi {}), buang {} non-Indonesia + {} duplikat, {:.0f}s".format(
        d["nama"], len(hasil), diambil_total_akumulasi + len(hasil), dibuang_bahasa, dibuang_duplikat, time.time() - t_mulai))
    return hasil


laporan_sumber = {}
hasil_per_dataset = {}
print("=== Panen Mode B: {} dataset ===".format(len(WHITELIST_MODE_B)))
for d in WHITELIST_MODE_B:
    hasil_per_dataset[d["slug"]] = panen_dataset(d, laporan_sumber)

In [ ]:
# @title 9. gzip + SHA256 + manifest per dataset (format PRD Sec7, field lang wajib)
import gzip, hashlib, collections, datetime

def tulis_jsonl_gz(path, records):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def komposisi_bahasa(records):
    c = collections.Counter(r.get("lang", "id") for r in records)
    total = sum(c.values()) or 1
    return {k: round(v / total, 4) for k, v in c.items()}

def buat_manifest(d, records):
    slug = d["slug"]
    path_gz = "/kaggle/working/{}.jsonl.gz".format(slug)
    tulis_jsonl_gz(path_gz, records)
    manifest = {
        "tag": "panen-" + slug,
        "status": "STAGING - belum masuk korpus kanonik, wajib review manual (PRD Sec5)",
        "sumberHuggingFace": d["hf_id"],
        "lisensi": d["lisensi"],
        "generatedAt": datetime.datetime.utcnow().strftime("%Y-%m-%d"),
        "totalDokumenSesiIni": len(records),
        "totalKataApprox": sum(len(r["text"].split()) for r in records),
        "format": "jsonl.gz",
        "fields": ["text", "source", "license", "url", "lang"],
        "komposisiBahasa": komposisi_bahasa(records),
        "sha256": sha256_file(path_gz) if records else None,
        "dedup": "hash SHA1 per-teks dalam sesi ini - lihat catatan jujur di akhir notebook",
    }
    path_manifest = "/kaggle/working/manifest-{}.json".format(slug)
    with open(path_manifest, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return path_gz, path_manifest, manifest

paket_manifest = {}
for d in WHITELIST_MODE_B:
    records = hasil_per_dataset.get(d["slug"]) or []
    if not records:
        continue
    paket_manifest[d["slug"]] = buat_manifest(d, records)
    print(d["slug"] + ":", paket_manifest[d["slug"]][2]["totalDokumenSesiIni"], "dokumen ->", paket_manifest[d["slug"]][0])

if not paket_manifest:
    print("Tidak ada dokumen terkumpul sesi ini (cek anggaran di sel 1 / HF_TOKEN untuk dataset gated).")

In [ ]:
# @title 10. Publikasikan ke Release STAGING (tag panen-<dataset>, bukan tag kanonik)
for slug, (path_gz, path_manifest, manifest) in paket_manifest.items():
    tag = "panen-" + slug
    judul = "Panen Dataset - {} ({} dokumen sesi ini)".format(slug, manifest["totalDokumenSesiIni"])
    print("Upload ke tag", tag, "...")
    up_gz = unggah_asset(tag, os.path.basename(path_gz), path_gz, title=judul)
    print("  data:", up_gz.get("browser_download_url"))
    up_man = unggah_asset(tag, os.path.basename(path_manifest), path_manifest, title=judul)
    print("  manifest:", up_man.get("browser_download_url"))

print()
print("SELESAI. Semua tag panen-* adalah STAGING, bukan korpus-<kategori>-bersih kanonik.")
print("Masuk korpus training resmi tetap butuh review manual (dedupe lintas-file + klasifikasi")
print("kategori K1/K2/K3) sesuai PRD-DATA-RELEASE.md Sec5 - hanya dirigen/Grok yang publish ke tag kanonik.")

In [ ]:
# @title 11. Laporan akhir panen (ringkasan per dataset)
laporan = {"modePanen": MODE_PANEN, "dataset": laporan_sumber}
print(json.dumps(laporan, ensure_ascii=False, indent=2))

## Catatan jujur

- **Tidak ada crawl situs** — `colab/panen-korpus.ipynb` (Mode A, ~20
  situs whitelist) dihapus karena terbukti rapuh (bergantung struktur
  HTML tiap situs, gampang patah kalau situs berubah). Notebook ini murni
  dataset aggregator lewat HuggingFace `datasets` streaming.
- **Dedup terbatas per sesi**: hash SHA1 per-teks dicek dalam satu sesi
  jalan (satu `Save & Run All`), TIDAK dipersist lintas sesi. Kalau kernel
  di-resume setelah quota habis, offset dataset dilanjut dari titik
  terakhir (lewat `progress.json` di Release) sehingga risiko duplikat
  antar-sesi kecil, tapi bukan nol — dedup penuh lintas-file/lintas-sesi
  tetap jadi bagian langkah review manual sebelum masuk korpus kanonik
  (PRD Sec5 poin 5b: "gabungkan, dedupe ulang lintas-file").
- **Id dataset HuggingFace** (MADLAD-400/OSCAR/CulturaX/Indo4B) adalah
  rujukan terbaik yang diketahui saat notebook ini ditulis — kalau salah
  satu berubah/dipindah di HF Hub, sel panen mencetak peringatan dan
  melewati dataset itu (tidak menghentikan seluruh panen).
- **MADLAD-400 id = lumbung utama**: dapat anggaran dokumen+menit 3x
  dataset lain (`PENGALI_LUMBUNG_UTAMA` di sel 1) karena CC-BY-4.0 jelas
  dan volumenya besar - prioritas sesuai instruksi.
- **Belum masuk korpus training**: semua tag `panen-<dataset>` adalah
  staging. Menggabungkannya ke `korpus-<kategori>-bersih` tetap butuh
  langkah manual sesuai `PRD-DATA-RELEASE.md` §5 - dokumen ini sendiri
  TIDAK diubah oleh notebook ini.